# Scraping Boston Sports Data

I scraped the official team websites for the home game schedules for the 2024 season for the Red Sox (Fenway Park), Celtics, and Bruins (TD Garden).

In [22]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import datetime
import os

# Red Sox

In [29]:
# Scrape MLB Stats API for  Red Sox 2024
url = "https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId=111&season=2024"

response = requests.get(url)
print(f"API Response Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    games_list = []

    # Map codes to names
    game_type_map = {
        'S': 'Preseason',  # Spring Training
        'E': 'Exhibition',
        'R': 'Regular Season',
        'F': 'Postseason',  # Wild Card
        'D': 'Postseason',  # Division Series
        'L': 'Postseason',  # League Championship Series
        'W': 'Postseason',  # World Series
    }

    for date_item in data.get('dates', []):
        game_date = date_item['date']
        for game in date_item.get('games', []):
            teams = game.get('teams', {})
            home_team = teams.get('home', {}).get('team', {}).get('name')
            away_team = teams.get('away', {}).get('team', {}).get('name')

            is_home = home_team == 'Boston Red Sox'
            opponent = away_team if is_home else home_team

            # Fetch the actual venue name from the API
            venue_name = game.get('venue', {}).get('name', 'Unknown')

            # Extract game type
            raw_game_type = game.get('gameType', 'R')
            game_type = game_type_map.get(raw_game_type, 'Regular Season')

            games_list.append({
                'date': game_date,
                'team': 'Red Sox',
                'opponent': opponent,
                'venue': venue_name,
                'is_home': is_home,
                'game_type': game_type,
            })

    redsox_df = pd.DataFrame(games_list)

    # Filter for home games at Fenway Park
    # Note: Spring Training home games are played in Fort Myers, FL (JetBlue Park)
    # Filtering by venue == 'Fenway Park'
    redsox_home = redsox_df[
        (redsox_df['is_home']) & (redsox_df['venue'] == 'Fenway Park')
    ].copy()

    print(f"Total 2024 Red Sox Home Games at Fenway: {len(redsox_home)}")
    print(redsox_home['game_type'].value_counts())

    # Create data directory
    os.makedirs('data', exist_ok=True)

    # Save to CSV
    redsox_home.to_csv('data/redsox_2024_home_games.csv', index=False)
    display(redsox_home.head())
else:
    print('Failed to pull from MLB API.')

API Response Status: 200
Total 2024 Red Sox Home Games at Fenway: 83
game_type
Regular Season    83
Name: count, dtype: int64


,date,team,opponent,venue,is_home,game_type
45,2024-04-09,Red Sox,Baltimore Orioles,Fenway Park,True,Regular Season
46,2024-04-10,Red Sox,Baltimore Orioles,Fenway Park,True,Regular Season
47,2024-04-11,Red Sox,Baltimore Orioles,Fenway Park,True,Regular Season
48,2024-04-12,Red Sox,Los Angeles Angels,Fenway Park,True,Regular Season
49,2024-04-13,Red Sox,Los Angeles Angels,Fenway Park,True,Regular Season


## Celtics

In [30]:
# ESPN API for Celtics
def get_celtics_schedule(season_year):
    season_type_map = {
        1: 'Preseason',
        2: 'Regular Season',
        3: 'Postseason',
        4: 'Play-In',
    }

    games = []

    # Include Preseason, Regular Season, Postseason, and Play-In 
    for st_id, st_name in season_type_map.items():
        url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/bos/schedule?season={season_year}&seasontype={st_id}'
        response = requests.get(url)

        if response.status_code != 200:
            continue

        events = response.json().get('events', [])

        for event in events:
            utc_time = pd.to_datetime(event['date'])
            local_time = utc_time.tz_convert('America/New_York')
            game_date = local_time.strftime('%Y-%m-%d')

            competition = event['competitions'][0]

            # Determine home or away
            competitors = competition['competitors']
            home_team = next(c for c in competitors if c['homeAway'] == 'home')
            away_team = next(c for c in competitors if c['homeAway'] == 'away')

            is_home = home_team['team']['displayName'] == 'Boston Celtics'
            opponent = (
                away_team['team']['displayName']
                if is_home
                else home_team['team']['displayName']
            )

            games.append({
                'date': game_date,
                'team': 'Celtics',
                'opponent': opponent,
                'venue': 'TD Garden' if is_home else 'Away',
                'is_home': is_home,
                'game_type': st_name,
            })

    return games


# Full 2024 calendar year (2023-2024 and 2024-2025 seasons)
celtics_games = get_celtics_schedule(2024) + get_celtics_schedule(2025)
celtics_df = pd.DataFrame(celtics_games)

# Filter for home games
celtics_home_2024 = (
    celtics_df[
        (celtics_df['is_home'])
        & (celtics_df['date'] >= '2024-01-01')
        & (celtics_df['date'] <= '2024-12-31')
    ]
    .drop_duplicates(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)

# Save to CSV
os.makedirs('data', exist_ok=True)
celtics_home_2024.to_csv('data/celtics_2024_home_games.csv', index=False)

print(f'Total 2024 Celtics Home Games found: {len(celtics_home_2024)}')
print(celtics_home_2024['game_type'].value_counts())
display(celtics_home_2024.head())

Total 2024 Celtics Home Games found: 58
game_type
Regular Season    44
Postseason        11
Preseason          3
Name: count, dtype: int64


,date,team,opponent,venue,is_home,game_type
0,2024-01-05,Celtics,Utah Jazz,TD Garden,True,Regular Season
1,2024-01-10,Celtics,Minnesota Timberwolves,TD Garden,True,Regular Season
2,2024-01-13,Celtics,Houston Rockets,TD Garden,True,Regular Season
3,2024-01-17,Celtics,San Antonio Spurs,TD Garden,True,Regular Season
4,2024-01-19,Celtics,Denver Nuggets,TD Garden,True,Regular Season


## Bruins 

In [ ]:
def get_bruins_schedule(season_year):
    season_type_map = {
        1: 'Preseason',
        2: 'Regular Season',
        3: 'Postseason'
    }

    games = []
    
    # Preseason, Regular Season, and Postseason
    for st_id, st_name in season_type_map.items():
        url = f"https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/teams/bos/schedule?season={season_year}&seasontype={st_id}"
        response = requests.get(url)

        if response.status_code != 200:
            continue

        events = response.json().get('events', [])

        for event in events:
            utc_time = pd.to_datetime(event['date'])
            local_time = utc_time.tz_convert('America/New_York')
            game_date = local_time.strftime('%Y-%m-%d')

            competition = event['competitions'][0]
            competitors = competition['competitors']

            home_team = next(c for c in competitors if c['homeAway'] == 'home')
            away_team = next(c for c in competitors if c['homeAway'] == 'away')

            is_home = (home_team['team']['displayName'] == 'Boston Bruins')
            opponent = away_team['team']['displayName'] if is_home else home_team['team']['displayName']

            games.append({
                'date': game_date,
                'team': 'Bruins',
                'opponent': opponent,
                'venue': 'TD Garden' if is_home else 'Away',
                'is_home': is_home,
                'game_type': st_name
            })

    return games

# Cover full 2024 calendar year across 2023-24 and 2024-25 seasons
bruins_games = get_bruins_schedule(2024) + get_bruins_schedule(2025)
bruins_df = pd.DataFrame(bruins_games)

# Filter for home games
bruins_home_2024 = bruins_df[
    (bruins_df['is_home']) & 
    (bruins_df['date'] >= '2024-01-01') & 
    (bruins_df['date'] <= '2024-12-31')
].drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)

# Save to CSV
os.makedirs('data', exist_ok=True)
bruins_home_2024.to_csv('data/bruins_2024_home_games.csv', index=False)

print(f"Total 2024 Bruins Home Games found: {len(bruins_home_2024)}")
print(bruins_home_2024['game_type'].value_counts())
display(bruins_home_2024.head())

Total 2024 Bruins Home Games found: 55
game_type
Regular Season    45
Postseason         7
Preseason          3
Name: count, dtype: int64


,date,team,opponent,venue,is_home,game_type
0,2024-01-04,Bruins,Pittsburgh Penguins,TD Garden,True,Regular Season
1,2024-01-06,Bruins,Tampa Bay Lightning,TD Garden,True,Regular Season
2,2024-01-15,Bruins,New Jersey Devils,TD Garden,True,Regular Season
3,2024-01-18,Bruins,Colorado Avalanche,TD Garden,True,Regular Season
4,2024-01-20,Bruins,Montreal Canadiens,TD Garden,True,Regular Season
